# Ordered Logistic Regression Results (FAIR²) Exploration with `mlcroissant`
This notebook provides a practical guide to loading and exploring the FAIR² dataset using the `mlcroissant` library, following the Croissant standard for interoperable ML datasets.

### Dataset Source
The dataset source is specified via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will retrieve the dataset's Croissant schema and register its record sets, fields, and metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Examine the record sets, fields, and column IDs defined in the Croissant schema. The following code lists all available record sets and their fields, referencing all entities by their `@id`s.

In [ ]:
# Explore available record sets and their fields by @id
print("Available record sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- record_set @id: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            print(f"    - field @id: {field['@id']}")
    if 'column' in record_set:
        columns = record_set['column']
        if isinstance(columns, dict):
            columns = [columns]
        print("  Columns:")
        for column in columns:
            print(f"    - column @id: {column['@id']}")
            if 'name' in column:
                print(f"      name: {column['name']}")
    print("")
# Optionally, list all available @id's in the schema for reference
# print("All available @id's in dataset (record sets, fields, columns):")
# for record_set in dataset.record_sets:
#     print(f"RecordSet @id: {record_set.get('@id', '<no @id>')}")
#     for field in record_set.get('field', []):
#         print(f"  Field @id: {field.get('@id', '<no @id>')}")

## 3. Data Extraction
Now, load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the data overview above.

Below, we load *all* record sets detected in the schema. Data is referenced by their record set `@id`.

In [ ]:
# Collect all record set @id's available in this dataset
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns from one example record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Now process a numeric field from a record set. The following steps demonstrate:
- Selecting a numeric field by `@id` for analysis
- Filtering records by a threshold
- Normalizing the numeric field
- Optionally grouping by a secondary field (`group_field` by `@id`)

**Replace the example field IDs below with those relevant to your dataset as needed, based on your overview above.**

In [ ]:
# Choose appropriate record set and field @id for numerical analysis
# (Replace these values with actual @ids from section 2 as available)
example_record_set_id = record_set_ids[0] if record_set_ids else None
# You'll need to update this to a real numeric field @id from the printed schema fields above
example_numeric_field_id = None
# Find a numeric column if available
if example_record_set_id:
    for col in dataframes[example_record_set_id].columns:
        if 'log_likelihood' in col or 'coef' in col or 'std' in col or 'pval' in col:
            example_numeric_field_id = col
            break

if example_numeric_field_id is None:
    print('No clear numeric field available; please update example_numeric_field_id as needed.')
else:
    threshold = 0  # Example threshold, update depending on field meaning
    df = dataframes[example_record_set_id]
    filtered_df = df[df[example_numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {example_numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{example_numeric_field_id}_normalized"] = (
        (filtered_df[example_numeric_field_id].astype(float) - filtered_df[example_numeric_field_id].astype(float).mean()) /
        filtered_df[example_numeric_field_id].astype(float).std()
    )
    print(f"Normalized {example_numeric_field_id} for filtered records:")
    print(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    group_field_id = None
    for col in df.columns:
        # Find a likely group field (e.g., 'ward', 'region', 'gender', etc.)
        if col.lower() in ['ward', 'region', 'gender', 'county']:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        print('No categorical group field available for grouping.')

## 5. Visualization
Visualize numeric field distribution or relationships. Replace `example_numeric_field_id` and `group_field_id` with valid `@id`s as needed.

In [ ]:
# Basic visualization using matplotlib for the chosen numeric field
import matplotlib.pyplot as plt

if example_numeric_field_id and example_record_set_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 5))
    df[example_numeric_field_id].astype(float).hist(bins=30)
    plt.title(f"Distribution of {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if grouped
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        df.boxplot(column=example_numeric_field_id, by=group_field_id, grid=False, figsize=(8,5))
        plt.title(f"{example_numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.show()
else:
    print("Please set appropriate field @ids to enable plotting.")

## 6. Conclusion
In this notebook, we loaded FAIR² dataset metadata and explored its record sets using the `mlcroissant` library, referencing all entities by their `@id`s. We demonstrated how to extract and process records for analysis—including basic normalization, filtering, grouping, and visualization by field or group. Update the notebook to refine your analysis as you explore new record sets, fields, and dataset characteristics revealed in the metadata.